This notebook is used to check the quality of the data, do some safety check plotting and anything additional stuff that's not super analysis focused

# Settings

In [ ]:
import mne
import pandas as pd
import os
import sys
import numpy as np
import matplotlib
from mne import read_labels_from_annot
import scipy
import tqdm
from pathlib import Path
import pickle
import glob



matplotlib.use('Qt5Agg')
mne.set_log_level(verbose=30)
import matplotlib.pyplot as plt

In [ ]:
expt = 'IdvDiff'
PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "project_config.json").is_file():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "project_config.json").is_file():
    raise FileNotFoundError("Could not find project_config.json above the notebook working directory")
sys.path.insert(0, str(PROJECT_DIR))
from project_config import get_path

PATH = get_path("project_root")
main_dir = get_path("data")
mri_dir = get_path("mri")
meg_dir = get_path("meg")
log_dir = get_path("logs")
fig_dir = get_path("figures")
stc_homedir = get_path("stc")


# get subject names as a list of directories
manual = ['R2549', 'R2378', 'R2614']
subjects = [d for d in os.listdir(meg_dir) if "R" in d and ".md" not in d and d not in manual]

n_subj = len(subjects)
print(n_subj)

# Create dictionaries to store subject variables
save_plot = True # settings for the evoked plots
overwrite = False # do not overwrite existing covariance etc
crop = False
photodiode_realignment = True

os.chdir(PATH)

from helperfunctions import get_raws, find_white_rectangle_photodiode, get_task_onsets, make_subj_df


## Other details

In [ ]:
mapping = {
    'scrambled_fast': 'WordList',
    'scrambled_slow': 'WordList',
    'sents_fast': 'SentList',
    'sents_slow': 'SentList',
    'intact_fast': 'Story',
    'intact_slow': 'Story'
}

# define conditions
conditions = ['scrambled_fast', 'scrambled_slow',
        'sents_fast', 'sents_slow',
        'intact_fast', 'intact_slow'] 

# set plot parameters
cond_fast = [cond for cond in conditions if 'fast' in cond]
cond_slow = [cond for cond in conditions if 'slow' in cond]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.copper(np.linspace(0, 1,3))) # have plot cycle through colors

# Calculate number of bad channels

In [ ]:
channels = []

for subj in subjects:
    bad_chan_path = meg_dir/subj/'cache'/'bad_channels.p'
    with (open(bad_chan_path, "rb")) as f:
        channels.append(len(pickle.load(f)))

chan_data = pd.DataFrame({'subj':subjects, 'n_channels':channels})
chan_data['n_channels'].describe()

# Calculate photodiode delay across participants

In [ ]:
# Initialize delays dictionary
delays = {}
for subj in subjects:
    #raw = mne.io.read_raw_fif(os.path.join(MEGPATH, subj, 'preproc', f'{subj}_{expt}_ica-raw.fif'), preload=True)

    print("Searching for white rectangle photodiode signal...")
    best_channel, delay_info, response_percentage = find_white_rectangle_photodiode(raw, trigger_channel='STI 014', max_delay_ms=150)
    if best_channel:
        print(f"\nBest photodiode channel found: {best_channel} \n")
        delays[subj] = {'delay':f"{delay_info[0]:.1f} ± {delay_info[1]:.1f} ms", 'Response rate':f'{response_percentage:.1f}%'}
                        
    else:
        print("No suitable photodiode channel found!")
        delays[subj] = 0

print(sorted(delays.items()))

# Get reaction time for task performance

### Subject loop that reads in the data and gets the task reaction time from the triggers

In [ ]:
subj_dfs = []
for subj in subjects:
    try:
        PATH = os.path.join(meg_dir,subj)
        raw = get_raws(PATH)  
    except:
        print(f"No file found: {f'{subj}_{expt}1_NR-raw.fif'}")
        continue
    print(f'Data for {subj} loaded...')
    events = mne.find_events(raw, consecutive=True, min_duration = 0.002)
    task_onsets = get_task_onsets(events)
    df = make_subj_df(task_onsets, subj)
    subj_dfs.append(df)
    
print('Done!')
pd.concat(subj_dfs,ignore_index=True).to_csv(log_dir+'/Subject_task_rts.csv')

# Create evoked and plot them

In [ ]:
# adapted from a larger subject loop
for subj in subjects:
    print(f'Creating errorbar erps for {subj}')
    cleandata = mne.read_epochs(os.path.join(meg_dir, subj, 'preproc', f'{subj}_{expt}_final-epo.fif'), preload=True)
    print('Data read in...')
    
    # select and define all of the conditions 
    trials = cleandata["Type=='Trial'"] # filter out the personal experience and task segments
    del cleandata
    
    # create the plot
    fig, axs = plt.subplots(nrows=2, ncols=1,figsize=(12, 8), sharey=True, squeeze=False)
    fig.subplots_adjust(wspace=0.1,hspace=0.5)
    
    # start the big loop
    for row, cond_group in enumerate([cond_slow, cond_fast]):            
        for condition in cond_group:
            print(condition)
            data = select_condition(trials, condition)
            cond = data.average() # average over conditions
            ev = cond.data.mean(0) # average over channels
            sem = scipy.stats.sem(cond.data,axis=0) # get SEM across channels for each timepoint
            cond.comment = condition

            axs[row,0].errorbar(cond.times*1000, ev, yerr=sem, label=mapping[condition], alpha=0.7)
    
            # Formatting subplots
            if 'fast' in condition:
                axs[row,0].set_xlim(-100,450)
            else:
                axs[row,0].set_xlim(-100,800)
            axs[row,0].axvline(x=0, color='k', linestyle='--')
            axs[row,0].set_title(f'{["Slow", "Fast"][row]}', fontsize=14)
            axs[row,0].set_ylabel('Mean evoked value')
            axs[row,0].set_xlabel('Time [ms]')
            axs[row,0].legend()
      
        
    del trials  
    print(f'Plot finished! \n')
    fig.suptitle(f'Evoked comparisons {subj}', fontsize=18)
    fig.tight_layout()
    fig.savefig(fig_dir+f'/Evoked Comparisons-{subj}')

plt.show()


# Check subject level STCs for rogue subject

In [ ]:
# more plotting specific variables
hemis = ['lh', 'rh']
labels = read_labels_from_annot(subject='fsaverage', parc='PALS_B12_Brodmann', subjects_dir=mri_dir)
ROIs = ['17']

In [ ]:
# read stcs, average activity within a given region, plot average on grid
os.environ["SUBJECTS_DIR"] = mri_dir

for subj in subjects:
    print(f'Creating stc errorbars for {subj}')
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 8), sharey=True, squeeze=False)
    fig.subplots_adjust(wspace=0.1,hspace=0.5)
    for BA in ROIs:
        for col, hemi in enumerate(hemis):
            for row, cond_group in enumerate([cond_slow, cond_fast]):            
                for condition in cond_group:
                    print(condition)
                    filename = os.path.join(stc_homedir, condition, f'{subj}_{condition}-sym_dspm-{hemi}.stc')
                    cond_stc = mne.read_source_estimate(filename)
                    print('loading in data...')
                    
                    # subset data to label of interest
                    current_label = next(label for label in labels if label.name == f'Brodmann.{BA}-{hemi}')
                    label_avg = cond_stc.in_label(current_label)
    
                    subj_avg = label_avg.data.mean(0) # average over source vertices
                    sem_src = scipy.stats.sem(label_avg.data,axis=0) # get SEM across sources for each timepoint
    
                    axs[row,col].errorbar(cond_stc.times*1000, subj_avg, yerr=sem_src,label=mapping[condition], alpha=0.5)
    
    
                # Formatting subplots
                if 'fast' in condition:
                    axs[row,col].set_xlim(-100,450)
                else:
                    axs[row,col].set_xlim(-100,800)
        
                axs[row,col].axvline(x=0, color='k', linestyle='--')
                axs[row,col].set_title(f'BA {BA} - {hemi} ({["Slow", "Fast"][row]})', fontsize=14)
                axs[row,col].set_ylabel('Mean dSPM Value')
                axs[row,col].set_xlabel('Time [ms]')
                axs[row,col].legend()
        
        print(f'Plot finished for BA {BA} for {subj} \n')
        fig.suptitle(f'Comparisons for Brodmann Area {BA} - {subj}', fontsize=18)
        fig.tight_layout()
        fig.savefig(f'fig_dir/safetycheck/Comparisons for Brodmann Area {BA}-{subj}-sym',dpi=300)

plt.show()

In [ ]:
cond_stc.plot(subject='fsaverage',hemi='split', cortex='low_contrast', background='white')

# Create averaged stc response across subjects

In [ ]:
# creating an average stc for illustration purposes
condition = 'intact_slow'
stcs = []
for subj in subjects:
    filename = os.path.join(stc_homedir, condition, f'{subj}_{condition}_dSPM')
    cond_stc = mne.read_source_estimate(filename)
    stcs.append(cond_stc)

mean_stc = sum(stcs) / len(stcs)

mean_stc.plot(subject='fsaverage',hemi='split', cortex='low_contrast', background='white', views=['lat','med'], surface='smoothwm', colorbar=False)